In [7]:
from pathlib import Path

import json
import time

import joblib
import numpy as np
import pandas as pd

current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

processed_data_path = (
    project_root
    / "data"
    / "processed"
    / "heart_disease_clean.csv"
)

comparison_report_path = (
    project_root
    / "reports"
    / "model_comparison.csv"
)

if not processed_data_path.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {processed_data_path}"
    )

if not comparison_report_path.exists():
    raise FileNotFoundError(
        f"Model comparison report not found: "
        f"{comparison_report_path}"
    )

df = pd.read_csv(processed_data_path)

comparison_results = pd.read_csv(
    comparison_report_path
)

target_column = "heart_disease"

if target_column not in df.columns:
    raise KeyError(
        f"Target column '{target_column}' was not found."
    )

if df[target_column].isna().any():
    raise ValueError(
        "The target column contains missing values."
    )

required_result_columns = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC-AUC",
    "PR-AUC"
]

missing_result_columns = [
    column
    for column in required_result_columns
    if column not in comparison_results.columns
]

if missing_result_columns:
    raise KeyError(
        f"Missing model-comparison columns: "
        f"{missing_result_columns}"
    )

if "Dataset" in comparison_results.columns:
    comparison_results = comparison_results[
        comparison_results["Dataset"] == "Testing"
    ].copy()

if comparison_results.empty:
    raise ValueError(
        "No testing model results were found "
        "in the comparison report."
    )

numeric_result_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC-AUC",
    "PR-AUC"
]

for column in numeric_result_columns:
    comparison_results[column] = pd.to_numeric(
        comparison_results[column],
        errors="raise"
    )

ranked_results = (
    comparison_results
    .sort_values(
        by=[
            "PR-AUC",
            "ROC-AUC",
            "Recall",
            "F1 Score"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

best_model_name = ranked_results.loc[
    0,
    "Model"
]

best_model_result = ranked_results.iloc[0]

supported_models = [
    "Logistic Regression",
    "Decision Tree",
    "Random Forest"
]

if best_model_name not in supported_models:
    raise ValueError(
        f"Unsupported selected model: {best_model_name}"
    )

X = df.drop(columns=[target_column])
y = df[target_column].astype("int64")

target_classes = sorted(
    y.unique().tolist()
)

if target_classes != [0, 1]:
    raise ValueError(
        f"Expected target classes [0, 1], "
        f"but found {target_classes}."
    )

print(
    "Dataset and comparison report loaded successfully"
)
print("Dataset shape:", df.shape)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Target classes:", target_classes)
print("Selected model:", best_model_name)
print(
    "Selected model PR-AUC:",
    round(
        float(best_model_result["PR-AUC"]),
        4
    )
)
print(
    "Selected model ROC-AUC:",
    round(
        float(best_model_result["ROC-AUC"]),
        4
    )
)

Dataset and comparison report loaded successfully
Dataset shape: (4238, 16)
Feature matrix shape: (4238, 15)
Target shape: (4238,)
Target classes: [0, 1]
Selected model: Logistic Regression
Selected model PR-AUC: 0.2937
Selected model ROC-AUC: 0.6952


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.tree import DecisionTreeClassifier

continuous_features = [
    "age",
    "cigarettes_per_day",
    "total_cholesterol",
    "systolic_bp",
    "diastolic_bp",
    "bmi",
    "heart_rate",
    "glucose"
]

binary_features = [
    "current_smoker",
    "bp_meds",
    "prevalent_stroke",
    "prevalent_hypertension",
    "diabetes"
]

categorical_features = [
    "gender",
    "education"
]

all_defined_features = (
    continuous_features
    + binary_features
    + categorical_features
)

missing_features = sorted(
    set(X.columns) - set(all_defined_features)
)

unexpected_features = sorted(
    set(all_defined_features) - set(X.columns)
)

duplicate_features = sorted({
    feature
    for feature in all_defined_features
    if all_defined_features.count(feature) > 1
})

if missing_features:
    raise ValueError(
        f"Features not assigned to a group: "
        f"{missing_features}"
    )

if unexpected_features:
    raise ValueError(
        f"Defined features not found in the dataset: "
        f"{unexpected_features}"
    )

if duplicate_features:
    raise ValueError(
        f"Features assigned more than once: "
        f"{duplicate_features}"
    )


def build_model_pipeline(
    model_name: str
) -> Pipeline:
    """
    Build an end-to-end preprocessing and
    classification pipeline.
    """

    continuous_steps = [
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]

    # Logistic Regression requires scaled
    # continuous features
    if model_name == "Logistic Regression":
        continuous_steps.append(
            (
                "scaler",
                StandardScaler()
            )
        )

    continuous_pipeline = Pipeline(
        steps=continuous_steps
    )

    binary_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            )
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "continuous",
                continuous_pipeline,
                continuous_features
            ),
            (
                "binary",
                binary_pipeline,
                binary_features
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )

    if model_name == "Logistic Regression":
        classifier = LogisticRegression(
            max_iter=1000,
            random_state=42
        )

    elif model_name == "Decision Tree":
        classifier = DecisionTreeClassifier(
            random_state=42
        )

    elif model_name == "Random Forest":
        classifier = RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )

    else:
        raise ValueError(
            f"Unsupported model: {model_name}"
        )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                classifier
            )
        ]
    )


final_pipeline = build_model_pipeline(
    best_model_name
)

print(
    "Final model pipeline created successfully"
)
print("Selected classifier:", best_model_name)
print(
    "Total input features:",
    len(all_defined_features)
)
print(
    "Pipeline steps:",
    list(final_pipeline.named_steps.keys())
)

Final model pipeline created successfully
Selected classifier: Logistic Regression
Total input features: 15
Pipeline steps: ['preprocessor', 'classifier']


In [9]:
training_start = time.perf_counter()

final_pipeline.fit(
    X,
    y
)

training_end = time.perf_counter()

final_training_time = (
    training_end - training_start
)

fitted_classifier = (
    final_pipeline
    .named_steps["classifier"]
)

fitted_preprocessor = (
    final_pipeline
    .named_steps["preprocessor"]
)

transformed_feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)

if len(transformed_feature_names) == 0:
    raise ValueError(
        "No transformed features were produced"
    )

print("Final pipeline trained successfully")
print("Model:", best_model_name)
print("Training rows:", len(X))
print(
    "Original input features:",
    X.shape[1]
)
print(
    "Transformed features:",
    len(transformed_feature_names)
)
print(
    f"Training time: "
    f"{final_training_time:.4f} seconds"
)
print(
    "Model classes:",
    fitted_classifier.classes_.tolist()
)

Final pipeline trained successfully
Model: Logistic Regression
Training rows: 4238
Original input features: 15
Transformed features: 19
Training time: 0.0440 seconds
Model classes: [0, 1]


In [10]:
from datetime import datetime, timezone

models_directory = (
    project_root
    / "models"
)

models_directory.mkdir(
    parents=True,
    exist_ok=True
)

model_path = (
    models_directory
    / "heart_disease_prediction_pipeline.joblib"
)

metadata_path = (
    models_directory
    / "heart_disease_model_metadata.json"
)

saved_files = joblib.dump(
    final_pipeline,
    model_path
)

if not model_path.exists():
    raise FileNotFoundError(
        "The model pipeline was not saved"
    )

model_metadata = {
    "model_version": "1.0.0",
    "selected_model": best_model_name,
    "target_column": target_column,
    "class_labels": {
        "0": "No heart disease",
        "1": "Heart disease"
    },
    "input_features": X.columns.tolist(),
    "continuous_features": continuous_features,
    "binary_features": binary_features,
    "categorical_features": categorical_features,
    "training_rows": int(len(X)),
    "training_columns": int(X.shape[1]),
    "transformed_feature_count": int(
        len(transformed_feature_names)
    ),
    "positive_class_rate": float(
        y.mean()
    ),
    "trained_on_complete_dataset": True,
    "selection_metric": "PR-AUC",
    "classification_threshold": 0.50,
    "selection_results": {
        "accuracy": float(
            best_model_result["Accuracy"]
        ),
        "precision": float(
            best_model_result["Precision"]
        ),
        "recall": float(
            best_model_result["Recall"]
        ),
        "f1_score": float(
            best_model_result["F1 Score"]
        ),
        "roc_auc": float(
            best_model_result["ROC-AUC"]
        ),
        "pr_auc": float(
            best_model_result["PR-AUC"]
        )
    },
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

metadata_path.write_text(
    json.dumps(
        model_metadata,
        indent=4
    ),
    encoding="utf-8"
)

if not metadata_path.exists():
    raise FileNotFoundError(
        "The model metadata was not saved"
    )

model_size_mb = (
    model_path.stat().st_size
    / (1024 * 1024)
)

print("Model pipeline saved successfully")
print(
    "Model path:",
    model_path.relative_to(project_root)
)
print(
    "Metadata path:",
    metadata_path.relative_to(project_root)
)
print(
    f"Model size: {model_size_mb:.4f} MB"
)

Model pipeline saved successfully
Model path: models\heart_disease_prediction_pipeline.joblib
Metadata path: models\heart_disease_model_metadata.json
Model size: 0.0054 MB


In [11]:
if not model_path.exists():
    raise FileNotFoundError(
        f"Saved model not found: {model_path}"
    )

loaded_pipeline = joblib.load(
    model_path
)

if not hasattr(
    loaded_pipeline,
    "named_steps"
):
    raise TypeError(
        "The loaded object is not a "
        "scikit-learn pipeline"
    )

validation_data = (
    X.head(20).copy()
)

original_predictions = (
    final_pipeline.predict(
        validation_data
    )
)

original_probabilities = (
    final_pipeline.predict_proba(
        validation_data
    )[:, 1]
)

loaded_predictions = (
    loaded_pipeline.predict(
        validation_data
    )
)

loaded_probabilities = (
    loaded_pipeline.predict_proba(
        validation_data
    )[:, 1]
)

predictions_match = np.array_equal(
    original_predictions,
    loaded_predictions
)

probabilities_match = np.allclose(
    original_probabilities,
    loaded_probabilities
)

if not predictions_match:
    raise AssertionError(
        "Loaded model predictions do not "
        "match the original predictions"
    )

if not probabilities_match:
    raise AssertionError(
        "Loaded model probabilities do not "
        "match the original probabilities"
    )

validation_summary = pd.DataFrame({
    "Original Prediction":
        original_predictions,
    "Loaded Prediction":
        loaded_predictions,
    "Original Probability":
        original_probabilities,
    "Loaded Probability":
        loaded_probabilities
})

print(
    "Saved model validation completed successfully"
)
print(
    "Predictions match:",
    predictions_match
)
print(
    "Probabilities match:",
    probabilities_match
)
print(
    "Loaded pipeline steps:",
    list(
        loaded_pipeline.named_steps.keys()
    )
)

display(
    validation_summary.head(10)
)

Saved model validation completed successfully
Predictions match: True
Probabilities match: True
Loaded pipeline steps: ['preprocessor', 'classifier']


,Original Prediction,Loaded Prediction,Original Probability,Loaded Probability
0,0,0,0.048279,0.048279
1,0,0,0.047453,0.047453
2,0,0,0.154841,0.154841
3,0,0,0.365076,0.365076
4,0,0,0.105365,0.105365
5,0,0,0.113781,0.113781
6,0,0,0.185482,0.185482
7,0,0,0.060280,0.060280
8,0,0,0.196479,0.196479
9,0,0,0.253545,0.253545


In [12]:
sample_patients = pd.DataFrame([
    {
        "gender": "female",
        "age": 35,
        "education": "graduate",
        "current_smoker": 0,
        "cigarettes_per_day": 0,
        "bp_meds": 0,
        "prevalent_stroke": 0,
        "prevalent_hypertension": 0,
        "diabetes": 0,
        "total_cholesterol": 190,
        "systolic_bp": 115,
        "diastolic_bp": 75,
        "bmi": 22.5,
        "heart_rate": 70,
        "glucose": 80
    },
    {
        "gender": "male",
        "age": 68,
        "education": "primary_school",
        "current_smoker": 1,
        "cigarettes_per_day": 20,
        "bp_meds": 1,
        "prevalent_stroke": 0,
        "prevalent_hypertension": 1,
        "diabetes": 1,
        "total_cholesterol": 280,
        "systolic_bp": 165,
        "diastolic_bp": 100,
        "bmi": 31.0,
        "heart_rate": 85,
        "glucose": 160
    },
    {
        "gender": "female",
        "age": 54,
        "education": np.nan,
        "current_smoker": 0,
        "cigarettes_per_day": 0,
        "bp_meds": np.nan,
        "prevalent_stroke": 0,
        "prevalent_hypertension": 1,
        "diabetes": 0,
        "total_cholesterol": np.nan,
        "systolic_bp": 145,
        "diastolic_bp": 90,
        "bmi": 28.0,
        "heart_rate": 75,
        "glucose": np.nan
    }
])

required_input_features = (
    X.columns.tolist()
)

missing_sample_features = sorted(
    set(required_input_features)
    - set(sample_patients.columns)
)

unexpected_sample_features = sorted(
    set(sample_patients.columns)
    - set(required_input_features)
)

if missing_sample_features:
    raise ValueError(
        f"Missing sample features: "
        f"{missing_sample_features}"
    )

if unexpected_sample_features:
    raise ValueError(
        f"Unexpected sample features: "
        f"{unexpected_sample_features}"
    )

sample_patients = sample_patients[
    required_input_features
]

sample_predictions = (
    loaded_pipeline.predict(
        sample_patients
    )
)

sample_probabilities = (
    loaded_pipeline.predict_proba(
        sample_patients
    )[:, 1]
)

prediction_results = pd.DataFrame({
    "Patient": [
        "Synthetic Patient 1",
        "Synthetic Patient 2",
        "Synthetic Patient 3"
    ],
    "Predicted Class":
        sample_predictions,
    "Prediction Label": np.where(
        sample_predictions == 1,
        "Heart Disease",
        "No Heart Disease"
    ),
    "Heart Disease Probability":
        sample_probabilities.round(4)
})

display(prediction_results)

print(
    "Synthetic predictions completed successfully"
)
print(
    "These predictions are for technical "
    "demonstration only"
)

,Patient,Predicted Class,Prediction Label,Heart Disease Probability
0,Synthetic Patient 1,0,No Heart Disease,0.0242
1,Synthetic Patient 2,1,Heart Disease,0.7529
2,Synthetic Patient 3,0,No Heart Disease,0.1455


Synthetic predictions completed successfully
These predictions are for technical demonstration only
